

**Technical Assignment: Emitter Identification and Geolocation**

Objective:
This script processes raw radio frequency (RF) observations collected by an
airborne receiver to identify and geolocate stationary radar emitters.

Methodology:
The approach is a two-stage process: Classification followed by Location Estimation.

1.  Classification (Who):
    The DBSCAN clustering algorithm is used to group individual RF pulses into
    clusters, where each cluster represents a unique emitter. The clustering is
    performed in a 4-dimensional feature space consisting of the signal's
    Frequency, Pulse Repetition Interval (PRI), Pulse Width (PW), and a
    normalized Timestamp. Including time helps separate signals that might
    be similar but are not active concurrently. The features are standardized
    to ensure that no single feature disproportionately influences the result.

2.  Parameter Selection (The Interactive Part):
    DBSCAN's most critical parameter, `eps`, is determined through a data-driven,
    analyst-in-the-loop workflow. The script first generates a diagnostic plot
    of `eps` vs. the number of clusters found. This allows the analyst to
    visually identify a stable "plateau" region and select a robust `eps`
    value, rather than relying on a guess.

3.  Location Estimation (Where):
    A robust hybrid method is used for geolocation. It first attempts a
    high-accuracy, non-linear optimization using SciPy to find the best-fit
    location based on spherical geometry. If this advanced solver fails to
    converge (often due to poor signal geometry), it automatically falls back
    to a more robust linear least-squares method on a UTM projection. This
    ensures a location estimate is always provided while prioritizing accuracy.

Workflow:
This script is designed to be run in two steps:
1.  First Run: Generates a plot (`eps_vs_clusters_plot.png`) to aid in `eps` selection.
2.  Second Run: The user updates the `CHOSEN_EPS` variable, and the script is run again to produce the final analysis and report.

**Observation_001 with removing DF_Q[1,2]**

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.interpolate import make_interp_spline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import pyproj
import datetime

# SECTION 1: FUNCTIONS

def cross_track_distance(point_lat, point_lon, start_lat, start_lon, bearing_deg):
    """
    Calculates the shortest distance (in meters) from a point to a great-circle
    path (a line of bearing), using numerically stable formulas.
    """
    R_earth = 6371000  # Earth radius in meters
    bearing_rad = np.radians(bearing_deg)

    # Convert all degree values to radians for trigonometric functions
    lat1, lon1 = np.radians(start_lat), np.radians(start_lon)
    lat3, lon3 = np.radians(point_lat), np.radians(point_lon)

    # Calculate angular distance between the aircraft and the candidate point
    arccos_input = np.sin(lat1) * np.sin(lat3) + np.cos(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    ang_dist_ac = np.arccos(np.clip(arccos_input, -1.0, 1.0)) # Clip for numerical stability

    # Calculate the bearing from the aircraft to the candidate point
    bearing_ac_rad = np.arctan2(
        np.sin(lon3 - lon1) * np.cos(lat3),
        np.cos(lat1) * np.sin(lat3) - np.sin(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    )

    delta_bearing = bearing_ac_rad - bearing_rad

    # Calculate the final cross-track distance
    arcsin_input = np.sin(ang_dist_ac) * np.sin(delta_bearing)
    d_xt = np.arcsin(np.clip(arcsin_input, -1.0, 1.0)) # Clip for numerical stability

    return R_earth * d_xt

def estimate_location_scipy(df_group, lat_col, lon_col, angle_col):
    """
    Finds the optimal emitter location using a non-linear solver (SciPy).
    This is the high-accuracy primary method.
    """
    if len(df_group) < 2: return None, None

    # Define an error function to minimize: the sum of squared distances to all lines of bearing.
    def error_function(x):
        candidate_lat, candidate_lon = x[0], x[1]
        distances = df_group.apply(lambda row: cross_track_distance(candidate_lat, candidate_lon, row[lat_col], row[lon_col], row[angle_col]), axis=1)
        return np.sum(distances**2)

    # Use the average location of observations as a smart initial guess
    initial_guess = [df_group[lat_col].mean(), df_group[lon_col].mean()]

    # Run the optimization solver
    result = minimize(error_function, initial_guess, method='Nelder-Mead')

    return (result.x[0], result.x[1]) if result.success else (None, None)

def estimate_location_utm(df_group, lat_col, lon_col, angle_col):
    """
    Estimates emitter location using a linear least-squares method on a UTM projection.
    This serves as a robust fallback method.
    """
    if len(df_group) < 2: return None, None

    # Determine the correct UTM zone from the average longitude
    avg_lon = df_group[lon_col].mean()
    utm_zone = int((avg_lon + 180) / 6) + 1
    wgs84, utm_crs = pyproj.CRS("EPSG:4326"), pyproj.CRS(f"EPSG:326{utm_zone}")

    # Convert observation coordinates to the more accurate UTM grid
    transformer_to_utm = pyproj.Transformer.from_crs(wgs84, utm_crs, always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(df_group[lon_col].values, df_group[lat_col].values)

    # Formulate and solve the linear system of equations
    angle_rad = np.radians(90 - df_group[angle_col])
    A = np.vstack([np.cos(angle_rad), np.sin(angle_rad)]).T
    b = np.sum(A * np.vstack([utm_x, utm_y]).T, axis=1)
    try:
        loc, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        # Convert the UTM result back to standard latitude/longitude
        transformer_to_wgs84 = pyproj.Transformer.from_crs(utm_crs, wgs84, always_xy=True)
        est_lon, est_lat = transformer_to_wgs84.transform(loc[0], loc[1])
        return est_lat, est_lon
    except np.linalg.LinAlgError:
        return None, None

def estimate_location_hybrid(df_group, lat_col, lon_col, angle_col):
    """
    Manages the location estimation, trying the high-accuracy SciPy method first
    and falling back to the UTM method if it fails.
    """
    est_lat, est_lon = estimate_location_scipy(df_group, lat_col, lon_col, angle_col)
    if est_lat is None:
        # This fallback makes the solution robust to poor signal geometry
        est_lat, est_lon = estimate_location_utm(df_group, lat_col, lon_col, angle_col)
    return est_lat, est_lon

def run_analysis():
    """Main function to run the full analysis workflow."""

    # --- Configuration ---
    INPUT_FILE = 'Observations_001.xlsx'
    # The user updates this value after the first run
    CHOSEN_EPS = 0.37
    MIN_SAMPLES = 20
    FILTER_LIST = [1, 2]
    # --- Define Column Names ---
    FREQ_COL, PRI_COL, PW_COL, TIME_COL = 'Freq', 'PRI', 'PW', 'Time'
    LAT_COL, LON_COL, ANGLE_COL, DFQ_COL = 'Lat', 'Lon', 'Angle', 'DF_Q'

    # --- Load and Scale Data ---
    print(f"Loading observations from {INPUT_FILE}...")
    obs_df = pd.read_excel(INPUT_FILE)

    # Convert time objects to a numerical format (seconds since midnight)
    def time_to_seconds(t):
        if isinstance(t, datetime.time):
            return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
        return np.nan
    if obs_df[TIME_COL].dtype == 'object':
        print("Converting 'Time' column from time objects to seconds...")
        obs_df[TIME_COL] = obs_df[TIME_COL].apply(time_to_seconds)

    obs_df[TIME_COL] = pd.to_numeric(obs_df[TIME_COL], errors='coerce')
    obs_df.dropna(subset=[TIME_COL], inplace=True)

    # Select and scale the four features for clustering
    features = obs_df[[FREQ_COL, PRI_COL, PW_COL, TIME_COL]]
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # --- Workflow Controller ---
    if CHOSEN_EPS == 0.0:
        # --- PART 1: Generate the `eps` vs. Number of Clusters Plot ---
        print("\n--- PART 1: Generating `eps` vs. Number of Clusters Plot ---")
        eps_range = np.arange(0.1, 1.01, 0.01)
        cluster_counts = []

        print("Testing a range of `eps` values to generate the diagnostic plot...")
        for eps in eps_range:
            dbscan = DBSCAN(eps=eps, min_samples=MIN_SAMPLES)
            clusters = dbscan.fit_predict(features_scaled)
            num_clusters = len(set(clusters) - {-1})
            cluster_counts.append(num_clusters)

        # Use spline interpolation to create a smooth curve for better visualization
        if len(eps_range) > 3:
            X_Y_Spline = make_interp_spline(eps_range, cluster_counts)
            X_ = np.linspace(eps_range.min(), eps_range.max(), 500)
            Y_ = X_Y_Spline(X_)
        else:
            X_ = eps_range
            Y_ = cluster_counts

        plt.figure(figsize=(12, 7))
        plt.plot(X_, Y_, label='Smoothed Trend')
        plt.scatter(eps_range, cluster_counts, color='red', zorder=5, s=10, label='Actual Data Points')
        plt.xticks(np.arange(0, 1.01, 0.05), rotation=90)
        plt.xlabel("Epsilon (`eps`) Value")
        plt.ylabel("Number of Clusters Found")
        plt.title(f"`eps` vs. Number of Clusters (for min_samples={MIN_SAMPLES})")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig("eps_vs_clusters_plot.png")

        print("\nACTION REQUIRED:")
        print("A plot named 'eps_vs_clusters_plot.png' has been saved.")
        print("1. Open the plot and find the 'elbow' or a stable 'plateau' region.")
        print("2. Choose an `eps` value from the start of this stable region.")
        print(f"3. Update the 'CHOSEN_EPS' variable in this script from {CHOSEN_EPS} to your new value.")
        print("4. Run the script again to get your final analysis.\n")
    else:
        # --- PART 2: Perform Final Analysis with Chosen EPS ---
        print(f"\n--- PART 2: Running Final Analysis with eps = {CHOSEN_EPS} ---")
        dbscan = DBSCAN(eps=CHOSEN_EPS, min_samples=MIN_SAMPLES)
        clusters = dbscan.fit_predict(features_scaled)
        obs_df.loc[features.index, 'Radar_ID'] = clusters

        print("\n--- Clustering Results ---")
        print(pd.Series(clusters).value_counts())
        obs_clustered_df = obs_df[obs_df['Radar_ID'] != -1].copy()
        num_radars_found = len(obs_clustered_df['Radar_ID'].unique())
        print(f"Found {num_radars_found} distinct radars.")

        results_list = []
        if num_radars_found > 0:
            print("\nCalculating parameters and estimating locations...")
            for radar_id in sorted(obs_clustered_df['Radar_ID'].unique()):
                full_group = obs_clustered_df[obs_clustered_df['Radar_ID'] == radar_id]
                filtered_group = full_group[~full_group[DFQ_COL].isin(FILTER_LIST)]
                avg_params = full_group[[FREQ_COL, PRI_COL, PW_COL]].mean()

                est_lat_all, est_lon_all = estimate_location_hybrid(full_group, LAT_COL, LON_COL, ANGLE_COL)
                est_lat_filtered, est_lon_filtered = estimate_location_hybrid(filtered_group, LAT_COL, LON_COL, ANGLE_COL)

                results_list.append({
                    'Radar_ID': radar_id, 'Obs_Count': len(full_group), 'Filtered_Count': len(filtered_group),
                    'Avg_Freq': avg_params[FREQ_COL], 'Avg_PRI': avg_params[PRI_COL], 'Avg_PW': avg_params[PW_COL],
                    'Est_Lat_All': est_lat_all, 'Est_Lon_All': est_lon_all,
                    'Est_Lat_Filtered': est_lat_filtered, 'Est_Lon_Filtered': est_lon_filtered
                })

        # --- Display Final Report ---
        print("\n-------------------------------------------")
        print(f"Final Report for {INPUT_FILE}")
        print("-------------------------------------------")
        results_df = pd.DataFrame(results_list)
        pd.set_option('display.width', 120)
        pd.set_option('display.max_columns', 11)
        pd.set_option('display.float_format', '{:.4f}'.format)
        print(results_df.to_string())

        output_filename = f"results_dbscan_time_{INPUT_FILE.split('.')[0]}.csv"
        results_df.to_csv(output_filename, index=False)
        print(f"\nResults have also been saved to '{output_filename}'")

if __name__ == "__main__":
    run_analysis()

Loading observations from Observations_001.xlsx...
Converting 'Time' column from time objects to seconds...

--- PART 2: Running Final Analysis with eps = 0.37 ---

--- Clustering Results ---
6    2753
0    2522
3    1000
2    1000
7     581
4     505
5     495
1     478
9     358
8     308
Name: count, dtype: int64
Found 10 distinct radars.

Calculating parameters and estimating locations...

-------------------------------------------
Final Report for Observations_001.xlsx
-------------------------------------------
   Radar_ID  Obs_Count  Filtered_Count  Avg_Freq   Avg_PRI  Avg_PW  Est_Lat_All  Est_Lon_All  Est_Lat_Filtered  Est_Lon_Filtered
0    0.0000       2522            1992 1825.1499 2388.1860  9.9022      21.4316      24.9550           21.4011           24.9507
1    1.0000        478             377 1841.4414 2772.4331 98.0000      21.9780      24.9058           21.9142           24.9207
2    2.0000       1000             814 1889.4800 1022.1760 30.0000      22.8829      24.8

**Observation_001 without removing DF_Q[1,2]**

In [22]:
def cross_track_distance(point_lat, point_lon, start_lat, start_lon, bearing_deg):
    """
    Calculates the shortest distance (in meters) from a point to a great-circle
    path (a line of bearing), using numerically stable formulas.
    """
    R_earth = 6371000  # Earth radius in meters
    bearing_rad = np.radians(bearing_deg)

    # Convert all degree values to radians for trigonometric functions
    lat1, lon1 = np.radians(start_lat), np.radians(start_lon)
    lat3, lon3 = np.radians(point_lat), np.radians(point_lon)

    # Calculate angular distance between the aircraft and the candidate point
    arccos_input = np.sin(lat1) * np.sin(lat3) + np.cos(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    ang_dist_ac = np.arccos(np.clip(arccos_input, -1.0, 1.0)) # Clip for numerical stability

    # Calculate the bearing from the aircraft to the candidate point
    bearing_ac_rad = np.arctan2(
        np.sin(lon3 - lon1) * np.cos(lat3),
        np.cos(lat1) * np.sin(lat3) - np.sin(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    )

    delta_bearing = bearing_ac_rad - bearing_rad

    # Calculate the final cross-track distance
    arcsin_input = np.sin(ang_dist_ac) * np.sin(delta_bearing)
    d_xt = np.arcsin(np.clip(arcsin_input, -1.0, 1.0)) # Clip for numerical stability

    return R_earth * d_xt

def estimate_location_scipy(df_group, lat_col, lon_col, angle_col):
    """
    Finds the optimal emitter location using a non-linear solver (SciPy).
    This is the high-accuracy primary method.
    """
    if len(df_group) < 2: return None, None

    # Define an error function to minimize: the sum of squared distances to all lines of bearing.
    def error_function(x):
        candidate_lat, candidate_lon = x[0], x[1]
        distances = df_group.apply(lambda row: cross_track_distance(candidate_lat, candidate_lon, row[lat_col], row[lon_col], row[angle_col]), axis=1)
        return np.sum(distances**2)

    # Use the average location of observations as a smart initial guess
    initial_guess = [df_group[lat_col].mean(), df_group[lon_col].mean()]

    # Run the optimization solver
    result = minimize(error_function, initial_guess, method='Nelder-Mead')

    return (result.x[0], result.x[1]) if result.success else (None, None)

def estimate_location_utm(df_group, lat_col, lon_col, angle_col):
    """
    Estimates emitter location using a linear least-squares method on a UTM projection.
    This serves as a robust fallback method.
    """
    if len(df_group) < 2: return None, None

    # Determine the correct UTM zone from the average longitude
    avg_lon = df_group[lon_col].mean()
    utm_zone = int((avg_lon + 180) / 6) + 1
    wgs84, utm_crs = pyproj.CRS("EPSG:4326"), pyproj.CRS(f"EPSG:326{utm_zone}")

    # Convert observation coordinates to the more accurate UTM grid
    transformer_to_utm = pyproj.Transformer.from_crs(wgs84, utm_crs, always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(df_group[lon_col].values, df_group[lat_col].values)

    # Formulate and solve the linear system of equations
    angle_rad = np.radians(90 - df_group[angle_col])
    A = np.vstack([np.cos(angle_rad), np.sin(angle_rad)]).T
    b = np.sum(A * np.vstack([utm_x, utm_y]).T, axis=1)
    try:
        loc, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        # Convert the UTM result back to standard latitude/longitude
        transformer_to_wgs84 = pyproj.Transformer.from_crs(utm_crs, wgs84, always_xy=True)
        est_lon, est_lat = transformer_to_wgs84.transform(loc[0], loc[1])
        return est_lat, est_lon
    except np.linalg.LinAlgError:
        return None, None

def estimate_location_hybrid(df_group, lat_col, lon_col, angle_col):
    """
    Manages the location estimation, trying the high-accuracy SciPy method first
    and falling back to the UTM method if it fails.
    """
    est_lat, est_lon = estimate_location_scipy(df_group, lat_col, lon_col, angle_col)
    if est_lat is None:
        # This fallback makes the solution robust to poor signal geometry
        est_lat, est_lon = estimate_location_utm(df_group, lat_col, lon_col, angle_col)
    return est_lat, est_lon

def run_analysis():
    """Main function to run the full analysis workflow."""

    # --- Configuration ---
    INPUT_FILE = 'Observations_001.xlsx'
    # The user updates this value after the first run
    CHOSEN_EPS = 0.37
    MIN_SAMPLES = 20
    FILTER_LIST = []
    # --- Define Column Names ---
    FREQ_COL, PRI_COL, PW_COL, TIME_COL = 'Freq', 'PRI', 'PW', 'Time'
    LAT_COL, LON_COL, ANGLE_COL, DFQ_COL = 'Lat', 'Lon', 'Angle', 'DF_Q'

    # --- Load and Scale Data ---
    print(f"Loading observations from {INPUT_FILE}...")
    obs_df = pd.read_excel(INPUT_FILE)

    # Convert time objects to a numerical format (seconds since midnight)
    def time_to_seconds(t):
        if isinstance(t, datetime.time):
            return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
        return np.nan
    if obs_df[TIME_COL].dtype == 'object':
        print("Converting 'Time' column from time objects to seconds...")
        obs_df[TIME_COL] = obs_df[TIME_COL].apply(time_to_seconds)

    obs_df[TIME_COL] = pd.to_numeric(obs_df[TIME_COL], errors='coerce')
    obs_df.dropna(subset=[TIME_COL], inplace=True)

    # Select and scale the four features for clustering
    features = obs_df[[FREQ_COL, PRI_COL, PW_COL, TIME_COL]]
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # --- Workflow Controller ---
    if CHOSEN_EPS == 0.0:
        # --- PART 1: Generate the `eps` vs. Number of Clusters Plot ---
        print("\n--- PART 1: Generating `eps` vs. Number of Clusters Plot ---")
        eps_range = np.arange(0.1, 1.01, 0.01)
        cluster_counts = []

        print("Testing a range of `eps` values to generate the diagnostic plot...")
        for eps in eps_range:
            dbscan = DBSCAN(eps=eps, min_samples=MIN_SAMPLES)
            clusters = dbscan.fit_predict(features_scaled)
            num_clusters = len(set(clusters) - {-1})
            cluster_counts.append(num_clusters)

        # Use spline interpolation to create a smooth curve for better visualization
        if len(eps_range) > 3:
            X_Y_Spline = make_interp_spline(eps_range, cluster_counts)
            X_ = np.linspace(eps_range.min(), eps_range.max(), 500)
            Y_ = X_Y_Spline(X_)
        else:
            X_ = eps_range
            Y_ = cluster_counts

        plt.figure(figsize=(12, 7))
        plt.plot(X_, Y_, label='Smoothed Trend')
        plt.scatter(eps_range, cluster_counts, color='red', zorder=5, s=10, label='Actual Data Points')
        plt.xticks(np.arange(0, 1.01, 0.05), rotation=90)
        plt.xlabel("Epsilon (`eps`) Value")
        plt.ylabel("Number of Clusters Found")
        plt.title(f"`eps` vs. Number of Clusters (for min_samples={MIN_SAMPLES})")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig("eps_vs_clusters_plot.png")

        print("\nACTION REQUIRED:")
        print("A plot named 'eps_vs_clusters_plot.png' has been saved.")
        print("1. Open the plot and find the 'elbow' or a stable 'plateau' region.")
        print("2. Choose an `eps` value from the start of this stable region.")
        print(f"3. Update the 'CHOSEN_EPS' variable in this script from {CHOSEN_EPS} to your new value.")
        print("4. Run the script again to get your final analysis.\n")
    else:
        # --- PART 2: Perform Final Analysis with Chosen EPS ---
        print(f"\n--- PART 2: Running Final Analysis with eps = {CHOSEN_EPS} ---")
        dbscan = DBSCAN(eps=CHOSEN_EPS, min_samples=MIN_SAMPLES)
        clusters = dbscan.fit_predict(features_scaled)
        obs_df.loc[features.index, 'Radar_ID'] = clusters

        print("\n--- Clustering Results ---")
        print(pd.Series(clusters).value_counts())
        obs_clustered_df = obs_df[obs_df['Radar_ID'] != -1].copy()
        num_radars_found = len(obs_clustered_df['Radar_ID'].unique())
        print(f"Found {num_radars_found} distinct radars.")

        results_list = []
        if num_radars_found > 0:
            print("\nCalculating parameters and estimating locations...")
            for radar_id in sorted(obs_clustered_df['Radar_ID'].unique()):
                full_group = obs_clustered_df[obs_clustered_df['Radar_ID'] == radar_id]
                filtered_group = full_group[~full_group[DFQ_COL].isin(FILTER_LIST)]
                avg_params = full_group[[FREQ_COL, PRI_COL, PW_COL]].mean()

                est_lat_all, est_lon_all = estimate_location_hybrid(full_group, LAT_COL, LON_COL, ANGLE_COL)
                est_lat_filtered, est_lon_filtered = estimate_location_hybrid(filtered_group, LAT_COL, LON_COL, ANGLE_COL)

                results_list.append({
                    'Radar_ID': radar_id, 'Obs_Count': len(full_group), 'Filtered_Count': len(filtered_group),
                    'Avg_Freq': avg_params[FREQ_COL], 'Avg_PRI': avg_params[PRI_COL], 'Avg_PW': avg_params[PW_COL],
                    'Est_Lat_All': est_lat_all, 'Est_Lon_All': est_lon_all,
                    'Est_Lat_Filtered': est_lat_filtered, 'Est_Lon_Filtered': est_lon_filtered
                })

        # --- Display Final Report ---
        print("\n-------------------------------------------")
        print(f"Final Report for {INPUT_FILE}")
        print("-------------------------------------------")
        results_df = pd.DataFrame(results_list)
        pd.set_option('display.width', 120)
        pd.set_option('display.max_columns', 11)
        pd.set_option('display.float_format', '{:.4f}'.format)
        print(results_df.to_string())

        output_filename = f"results_dbscan_time_{INPUT_FILE.split('.')[0]}.csv"
        results_df.to_csv(output_filename, index=False)
        print(f"\nResults have also been saved to '{output_filename}'")

if __name__ == "__main__":
    run_analysis()

Loading observations from Observations_001.xlsx...
Converting 'Time' column from time objects to seconds...

--- PART 2: Running Final Analysis with eps = 0.37 ---

--- Clustering Results ---
6    2753
0    2522
3    1000
2    1000
7     581
4     505
5     495
1     478
9     358
8     308
Name: count, dtype: int64
Found 10 distinct radars.

Calculating parameters and estimating locations...

-------------------------------------------
Final Report for Observations_001.xlsx
-------------------------------------------
   Radar_ID  Obs_Count  Filtered_Count  Avg_Freq   Avg_PRI  Avg_PW  Est_Lat_All  Est_Lon_All  Est_Lat_Filtered  Est_Lon_Filtered
0    0.0000       2522            2522 1825.1499 2388.1860  9.9022      21.4316      24.9550           21.4316           24.9550
1    1.0000        478             478 1841.4414 2772.4331 98.0000      21.9780      24.9058           21.9780           24.9058
2    2.0000       1000            1000 1889.4800 1022.1760 30.0000      22.8829      24.8

**Calculating error of Observation_001 with removing DF_Q[1,2]**

In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# Helper function to calculate distance between two lat/lon points in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371 # Earth radius in kilometers
    if any(pd.isna([lat1, lon1, lat2, lon2])): return np.nan
    dLat, dLon, lat1, lat2 = map(np.radians, [lat2 - lat1, lon2 - lon1, lat1, lat2])
    a = np.sin(dLat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dLon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Helper function to parse the messy ground truth values
def parse_value(value_str):
    value_str = str(value_str).strip()
    if '-' in value_str:
        parts = [float(p) for p in value_str.split('-') if p]
        return np.mean(parts) if parts else np.nan
    elif ',' in value_str:
        parts = [float(p.strip()) for p in value_str.split(',') if p]
        return np.mean(parts) if parts else np.nan
    else:
        return float(value_str)

# --- 1. Your Estimated Data ---
estimated_data = {
    'Radar_ID': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    'Avg_Freq': [1825.1499, 1841.4414, 1889.4800, 1923.3500, 1933.8673, 1933.7838, 2015.9622, 2064.5697, 2086.0325, 2085.8240],
    'Avg_PRI': [2388.1860, 2772.4331, 1022.1760, 905.8900, 1666.0000, 2222.0000, 2133.0272, 1655.7969, 3333.0000, 2222.0000],
    'Avg_PW': [9.9022, 98.0000, 30.0000, 75.0000, 2.4000, 2.4000, 6.7722, 3.6898, 2.4000, 2.4000],
    'Est_Lat_Filtered': [21.4011, 21.9142, 22.8640, 24.0680, 23.6626, 23.6187, 27.9568, 28.7086, 29.5131, 29.4943],
    'Est_Lon_Filtered': [24.9507, 24.9207, 24.7998, 25.0019, 25.9732, 26.0062, 25.0782, 25.0476, 25.9937, 25.8733]
}
estimated_df = pd.DataFrame(estimated_data)

# --- 2. Raw Ground Truth Data ---
ground_truth_raw = {
    'Sno': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Frequency': ['1800-1815', '1825-1843', '1827-1856', '1883-1896', '1912-1935', '1928-1940', '1980-2014', '2010-2030', '2032-2039', '2072-2100'],
    'PRI': ['2000-2010', '2281, 2419, 2654, 3007', '2084, 2267, 2419, 2654, 2890, 3056, 3275, 3444', '990, 1024, 1054', '872, 904, 946', '1666, 2222', '2000-2200', '2099', '1642, 1987, 2265, 2400', '1666, 2222, 3333'],
    'PW': ['10-15', '2.6, 6.5, 12.7', '10.2, 98', '30', '75', '2.4', '1.5-6.5', '10.6', '1.5, 9.4', '2.4'],
    'Latitude': [19.8775, 20.4273, 21.5086, 24.4546, 25.0335, 26.1642, 26.1381, 27.2327, 28.3128, 29.4965],
    'Longitude': [27.6946, 27.833, 28.1758, 22.631, 21.4268, 26.6978, 23.6046, 23.8347, 22.2464, 26.8206]
}
ground_truth_df = pd.DataFrame(ground_truth_raw)

# --- 3. Clean the Ground Truth Data ---
ground_truth_df['Frequency'] = ground_truth_df['Frequency'].apply(parse_value)
ground_truth_df['PRI'] = ground_truth_df['PRI'].apply(parse_value)
ground_truth_df['PW'] = ground_truth_df['PW'].apply(parse_value)

# --- 4. Match Clusters using the Hungarian Algorithm and Calculate Error ---
results = []
scaler = StandardScaler()

est_rf_features = estimated_df[['Avg_Freq', 'Avg_PRI', 'Avg_PW']]
gt_rf_features = ground_truth_df[['Frequency', 'PRI', 'PW']]

# Fit the scaler on the estimated data
est_rf_scaled = scaler.fit_transform(est_rf_features)

# --- FIX: Temporarily rename GT columns to match before transforming ---
gt_rf_features.columns = est_rf_features.columns
gt_rf_scaled = scaler.transform(gt_rf_features)
# --- End of Fix ---

# Create a cost matrix where cost is the distance between RF parameters
cost_matrix = cdist(est_rf_scaled, gt_rf_scaled)
# Find the optimal assignment that minimizes the total cost
est_indices, gt_indices = linear_sum_assignment(cost_matrix)

# Reorder the dataframes to align with the optimal matching
matched_estimated_df = estimated_df.iloc[est_indices].reset_index(drop=True)
matched_ground_truth_df = ground_truth_df.iloc[gt_indices].reset_index(drop=True)

# Combine the matched dataframes
final_comparison_df = pd.concat([matched_estimated_df, matched_ground_truth_df], axis=1)

# Calculate the location error for the matched pairs
final_comparison_df['Location_Error_km'] = final_comparison_df.apply(
    lambda row: haversine(
        row['Est_Lat_Filtered'], row['Est_Lon_Filtered'],
        row['Latitude'], row['Longitude']
    ),
    axis=1
)

# --- 5. Display the Final Comparison Table ---
pd.set_option('display.float_format', '{:.4f}'.format)
print("--- Final Comparison with 1-to-1 Matching and Location Error ---")

report_columns = [
    'Radar_ID', 'Sno', 'Avg_Freq', 'Frequency', 'Avg_PRI', 'PRI', 'Avg_PW', 'PW',
    'Est_Lat_Filtered', 'Latitude', 'Est_Lon_Filtered', 'Longitude', 'Location_Error_km'
]
print(final_comparison_df[report_columns].to_string())

# Calculate and print the average error
average_error = final_comparison_df['Location_Error_km'].mean()
print(f"\nAverage Location Error: {average_error:.2f} km")

--- Final Comparison with 1-to-1 Matching and Location Error ---
   Radar_ID  Sno  Avg_Freq  Frequency   Avg_PRI       PRI  Avg_PW      PW  Est_Lat_Filtered  Latitude  Est_Lon_Filtered  Longitude  Location_Error_km
0         0    2 1825.1499  1834.0000 2388.1860 2590.2500  9.9022  7.2667           21.4011   20.4273           24.9507    27.8330           318.3531
1         1    3 1841.4414  1841.5000 2772.4331 2761.1250 98.0000 54.1000           21.9142   21.5086           24.9207    28.1758           339.2772
2         2    4 1889.4800  1889.5000 1022.1760 1022.6667 30.0000 30.0000           22.8640   24.4546           24.7998    22.6310           282.9632
3         3    5 1923.3500  1923.5000  905.8900  907.3333 75.0000 75.0000           24.0680   25.0335           25.0019    21.4268           377.1782
4         4    6 1933.8673  1934.0000 1666.0000 1944.0000  2.4000  2.4000           23.6626   26.1642           25.9732    26.6978           287.6010
5         5    1 1933.7838  1807.50

**Calculating error of Observation_001 without removing DF_Q[1,2]**

In [23]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# Helper function to calculate distance between two lat/lon points in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371 # Earth radius in kilometers
    if any(pd.isna([lat1, lon1, lat2, lon2])): return np.nan
    dLat, dLon, lat1, lat2 = map(np.radians, [lat2 - lat1, lon2 - lon1, lat1, lat2])
    a = np.sin(dLat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dLon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Helper function to parse the messy ground truth values
def parse_value(value_str):
    value_str = str(value_str).strip()
    if '-' in value_str:
        parts = [float(p) for p in value_str.split('-') if p]
        return np.mean(parts) if parts else np.nan
    elif ',' in value_str:
        parts = [float(p.strip()) for p in value_str.split(',') if p]
        return np.mean(parts) if parts else np.nan
    else:
        return float(value_str)

# --- 1. Your NEW Estimated Data ---
estimated_data = {
    'Radar_ID': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    'Obs_Count': [2522, 478, 1000, 1000, 505, 495, 2753, 581, 308, 358],
    'Filtered_Count': [2522, 478, 1000, 1000, 505, 495, 2753, 581, 308, 296], # Note: Last value was cut off, using 296 from a previous run
    'Avg_Freq': [1825.1499, 1841.4414, 1889.4800, 1923.3500, 1933.8673, 1933.7838, 2015.9622, 2064.5697, 2086.0325, 2085.8240],
    'Avg_PRI': [2388.1860, 2772.4331, 1022.1760, 905.8900, 1666.0000, 2222.0000, 2133.0272, 1655.7969, 3333.0000, 2222.0000],
    'Avg_PW': [9.9022, 98.0000, 30.0000, 75.0000, 2.4000, 2.4000, 6.7722, 3.6898, 2.4000, 2.4000],
    'Est_Lat_Filtered': [21.4316, 21.9780, 22.8829, 24.1103, 25.7541, 25.7667, 27.8192, 28.7332, 29.4902, 28.5684],
    'Est_Lon_Filtered': [24.9550, 24.9058, 24.8631, 25.0470, 25.9645, 25.9540, 24.5586, 25.0458, 26.0596, 25.0612]
}
estimated_df = pd.DataFrame(estimated_data)

# --- 2. Raw Ground Truth Data ---
ground_truth_raw = {
    'Sno': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Frequency': ['1800-1815', '1825-1843', '1827-1856', '1883-1896', '1912-1935', '1928-1940', '1980-2014', '2010-2030', '2032-2039', '2072-2100'],
    'PRI': ['2000-2010', '2281, 2419, 2654, 3007', '2084, 2267, 2419, 2654, 2890, 3056, 3275, 3444', '990, 1024, 1054', '872, 904, 946', '1666, 2222', '2000-2200', '2099', '1642, 1987, 2265, 2400', '1666, 2222, 3333'],
    'PW': ['10-15', '2.6, 6.5, 12.7', '10.2, 98', '30', '75', '2.4', '1.5-6.5', '10.6', '1.5, 9.4', '2.4'],
    'Latitude': [19.8775, 20.4273, 21.5086, 24.4546, 25.0335, 26.1642, 26.1381, 27.2327, 28.3128, 29.4965],
    'Longitude': [27.6946, 27.833, 28.1758, 22.631, 21.4268, 26.6978, 23.6046, 23.8347, 22.2464, 26.8206]
}
ground_truth_df = pd.DataFrame(ground_truth_raw)

# --- 3. Clean the Ground Truth Data ---
ground_truth_df['Frequency'] = ground_truth_df['Frequency'].apply(parse_value)
ground_truth_df['PRI'] = ground_truth_df['PRI'].apply(parse_value)
ground_truth_df['PW'] = ground_truth_df['PW'].apply(parse_value)

# --- 4. Match Clusters using the Hungarian Algorithm and Calculate Error ---
scaler = StandardScaler()
est_rf_features = estimated_df[['Avg_Freq', 'Avg_PRI', 'Avg_PW']]
gt_rf_features = ground_truth_df[['Frequency', 'PRI', 'PW']]

est_rf_scaled = scaler.fit_transform(est_rf_features)

# Temporarily rename GT columns to match before transforming
gt_rf_features.columns = est_rf_features.columns
gt_rf_scaled = scaler.transform(gt_rf_features)

# Create a cost matrix and find the optimal 1-to-1 assignment
cost_matrix = cdist(est_rf_scaled, gt_rf_scaled)
est_indices, gt_indices = linear_sum_assignment(cost_matrix)

# Reorder the dataframes to align with the optimal matching
matched_estimated_df = estimated_df.iloc[est_indices].reset_index(drop=True)
matched_ground_truth_df = ground_truth_df.iloc[gt_indices].reset_index(drop=True)

# Combine the matched dataframes
final_comparison_df = pd.concat([matched_estimated_df, matched_ground_truth_df], axis=1)

# Calculate the location error for the matched pairs
final_comparison_df['Location_Error_km'] = final_comparison_df.apply(
    lambda row: haversine(
        row['Est_Lat_Filtered'], row['Est_Lon_Filtered'],
        row['Latitude'], row['Longitude']
    ),
    axis=1
)

# --- 5. Display the Final Comparison Table ---
pd.set_option('display.float_format', '{:.4f}'.format)
print("--- Final Comparison with 1-to-1 Matching and Location Error ---")
report_columns = [
    'Radar_ID', 'Sno', 'Avg_Freq', 'Frequency', 'Avg_PRI', 'PRI', 'Avg_PW', 'PW',
    'Est_Lat_Filtered', 'Latitude', 'Est_Lon_Filtered', 'Longitude', 'Location_Error_km'
]
print(final_comparison_df[report_columns].to_string())

# Calculate and print the average error
average_error = final_comparison_df['Location_Error_km'].mean()
print(f"\nAverage Location Error: {average_error:.2f} km")

--- Final Comparison with 1-to-1 Matching and Location Error ---
   Radar_ID  Sno  Avg_Freq  Frequency   Avg_PRI       PRI  Avg_PW      PW  Est_Lat_Filtered  Latitude  Est_Lon_Filtered  Longitude  Location_Error_km
0         0    2 1825.1499  1834.0000 2388.1860 2590.2500  9.9022  7.2667           21.4316   20.4273           24.9550    27.8330           319.0753
1         1    3 1841.4414  1841.5000 2772.4331 2761.1250 98.0000 54.1000           21.9780   21.5086           24.9058    28.1758           341.7396
2         2    4 1889.4800  1889.5000 1022.1760 1022.6667 30.0000 30.0000           22.8829   24.4546           24.8631    22.6310           286.7248
3         3    5 1923.3500  1923.5000  905.8900  907.3333 75.0000 75.0000           24.1103   25.0335           25.0470    21.4268           380.1972
4         4    6 1933.8673  1934.0000 1666.0000 1944.0000  2.4000  2.4000           25.7541   26.1642           25.9645    26.6978            86.3372
5         5    1 1933.7838  1807.50

**Observation_011 with removing DF_Q[1,2]**
*prediction only*

In [25]:
def cross_track_distance(point_lat, point_lon, start_lat, start_lon, bearing_deg):
    """
    Calculates the shortest distance (in meters) from a point to a great-circle
    path (a line of bearing), using numerically stable formulas.
    """
    R_earth = 6371000  # Earth radius in meters
    bearing_rad = np.radians(bearing_deg)

    # Convert all degree values to radians for trigonometric functions
    lat1, lon1 = np.radians(start_lat), np.radians(start_lon)
    lat3, lon3 = np.radians(point_lat), np.radians(point_lon)

    # Calculate angular distance between the aircraft and the candidate point
    arccos_input = np.sin(lat1) * np.sin(lat3) + np.cos(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    ang_dist_ac = np.arccos(np.clip(arccos_input, -1.0, 1.0)) # Clip for numerical stability

    # Calculate the bearing from the aircraft to the candidate point
    bearing_ac_rad = np.arctan2(
        np.sin(lon3 - lon1) * np.cos(lat3),
        np.cos(lat1) * np.sin(lat3) - np.sin(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    )

    delta_bearing = bearing_ac_rad - bearing_rad

    # Calculate the final cross-track distance
    arcsin_input = np.sin(ang_dist_ac) * np.sin(delta_bearing)
    d_xt = np.arcsin(np.clip(arcsin_input, -1.0, 1.0)) # Clip for numerical stability

    return R_earth * d_xt

def estimate_location_scipy(df_group, lat_col, lon_col, angle_col):
    """
    Finds the optimal emitter location using a non-linear solver (SciPy).
    This is the high-accuracy primary method.
    """
    if len(df_group) < 2: return None, None

    # Define an error function to minimize: the sum of squared distances to all lines of bearing.
    def error_function(x):
        candidate_lat, candidate_lon = x[0], x[1]
        distances = df_group.apply(lambda row: cross_track_distance(candidate_lat, candidate_lon, row[lat_col], row[lon_col], row[angle_col]), axis=1)
        return np.sum(distances**2)

    # Use the average location of observations as a smart initial guess
    initial_guess = [df_group[lat_col].mean(), df_group[lon_col].mean()]

    # Run the optimization solver
    result = minimize(error_function, initial_guess, method='Nelder-Mead')

    return (result.x[0], result.x[1]) if result.success else (None, None)

def estimate_location_utm(df_group, lat_col, lon_col, angle_col):
    """
    Estimates emitter location using a linear least-squares method on a UTM projection.
    This serves as a robust fallback method.
    """
    if len(df_group) < 2: return None, None

    # Determine the correct UTM zone from the average longitude
    avg_lon = df_group[lon_col].mean()
    utm_zone = int((avg_lon + 180) / 6) + 1
    wgs84, utm_crs = pyproj.CRS("EPSG:4326"), pyproj.CRS(f"EPSG:326{utm_zone}")

    # Convert observation coordinates to the more accurate UTM grid
    transformer_to_utm = pyproj.Transformer.from_crs(wgs84, utm_crs, always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(df_group[lon_col].values, df_group[lat_col].values)

    # Formulate and solve the linear system of equations
    angle_rad = np.radians(90 - df_group[angle_col])
    A = np.vstack([np.cos(angle_rad), np.sin(angle_rad)]).T
    b = np.sum(A * np.vstack([utm_x, utm_y]).T, axis=1)
    try:
        loc, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        # Convert the UTM result back to standard latitude/longitude
        transformer_to_wgs84 = pyproj.Transformer.from_crs(utm_crs, wgs84, always_xy=True)
        est_lon, est_lat = transformer_to_wgs84.transform(loc[0], loc[1])
        return est_lat, est_lon
    except np.linalg.LinAlgError:
        return None, None

def estimate_location_hybrid(df_group, lat_col, lon_col, angle_col):
    """
    Manages the location estimation, trying the high-accuracy SciPy method first
    and falling back to the UTM method if it fails.
    """
    est_lat, est_lon = estimate_location_scipy(df_group, lat_col, lon_col, angle_col)
    if est_lat is None:
        # This fallback makes the solution robust to poor signal geometry
        est_lat, est_lon = estimate_location_utm(df_group, lat_col, lon_col, angle_col)
    return est_lat, est_lon

def run_analysis():
    """Main function to run the full analysis workflow."""

    # --- Configuration ---
    INPUT_FILE = 'Observations_011.xlsx'
    # The user updates this value after the first run
    CHOSEN_EPS = 0.55
    MIN_SAMPLES = 20
    FILTER_LIST = [1,2]
    # --- Define Column Names ---
    FREQ_COL, PRI_COL, PW_COL, TIME_COL = 'Freq', 'PRI', 'PW', 'Time'
    LAT_COL, LON_COL, ANGLE_COL, DFQ_COL = 'Lat', 'Lon', 'Angle', 'DF_Q'

    # --- Load and Scale Data ---
    print(f"Loading observations from {INPUT_FILE}...")
    obs_df = pd.read_excel(INPUT_FILE)

    # Convert time objects to a numerical format (seconds since midnight)
    def time_to_seconds(t):
        if isinstance(t, datetime.time):
            return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
        return np.nan
    if obs_df[TIME_COL].dtype == 'object':
        print("Converting 'Time' column from time objects to seconds...")
        obs_df[TIME_COL] = obs_df[TIME_COL].apply(time_to_seconds)

    obs_df[TIME_COL] = pd.to_numeric(obs_df[TIME_COL], errors='coerce')
    obs_df.dropna(subset=[TIME_COL], inplace=True)

    # Select and scale the four features for clustering
    features = obs_df[[FREQ_COL, PRI_COL, PW_COL, TIME_COL]]
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # --- Workflow Controller ---
    if CHOSEN_EPS == 0.0:
        # --- PART 1: Generate the `eps` vs. Number of Clusters Plot ---
        print("\n--- PART 1: Generating `eps` vs. Number of Clusters Plot ---")
        eps_range = np.arange(0.1, 1.01, 0.01)
        cluster_counts = []

        print("Testing a range of `eps` values to generate the diagnostic plot...")
        for eps in eps_range:
            dbscan = DBSCAN(eps=eps, min_samples=MIN_SAMPLES)
            clusters = dbscan.fit_predict(features_scaled)
            num_clusters = len(set(clusters) - {-1})
            cluster_counts.append(num_clusters)

        # Use spline interpolation to create a smooth curve for better visualization
        if len(eps_range) > 3:
            X_Y_Spline = make_interp_spline(eps_range, cluster_counts)
            X_ = np.linspace(eps_range.min(), eps_range.max(), 500)
            Y_ = X_Y_Spline(X_)
        else:
            X_ = eps_range
            Y_ = cluster_counts

        plt.figure(figsize=(12, 7))
        plt.plot(X_, Y_, label='Smoothed Trend')
        plt.scatter(eps_range, cluster_counts, color='red', zorder=5, s=10, label='Actual Data Points')
        plt.xticks(np.arange(0, 1.01, 0.05), rotation=90)
        plt.xlabel("Epsilon (`eps`) Value")
        plt.ylabel("Number of Clusters Found")
        plt.title(f"`eps` vs. Number of Clusters (for min_samples={MIN_SAMPLES})")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig("eps_vs_clusters_plot.png")

        print("\nACTION REQUIRED:")
        print("A plot named 'eps_vs_clusters_plot.png' has been saved.")
        print("1. Open the plot and find the 'elbow' or a stable 'plateau' region.")
        print("2. Choose an `eps` value from the start of this stable region.")
        print(f"3. Update the 'CHOSEN_EPS' variable in this script from {CHOSEN_EPS} to your new value.")
        print("4. Run the script again to get your final analysis.\n")
    else:
        # --- PART 2: Perform Final Analysis with Chosen EPS ---
        print(f"\n--- PART 2: Running Final Analysis with eps = {CHOSEN_EPS} ---")
        dbscan = DBSCAN(eps=CHOSEN_EPS, min_samples=MIN_SAMPLES)
        clusters = dbscan.fit_predict(features_scaled)
        obs_df.loc[features.index, 'Radar_ID'] = clusters

        print("\n--- Clustering Results ---")
        print(pd.Series(clusters).value_counts())
        obs_clustered_df = obs_df[obs_df['Radar_ID'] != -1].copy()
        num_radars_found = len(obs_clustered_df['Radar_ID'].unique())
        print(f"Found {num_radars_found} distinct radars.")

        results_list = []
        if num_radars_found > 0:
            print("\nCalculating parameters and estimating locations...")
            for radar_id in sorted(obs_clustered_df['Radar_ID'].unique()):
                full_group = obs_clustered_df[obs_clustered_df['Radar_ID'] == radar_id]
                filtered_group = full_group[~full_group[DFQ_COL].isin(FILTER_LIST)]
                avg_params = full_group[[FREQ_COL, PRI_COL, PW_COL]].mean()

                est_lat_all, est_lon_all = estimate_location_hybrid(full_group, LAT_COL, LON_COL, ANGLE_COL)
                est_lat_filtered, est_lon_filtered = estimate_location_hybrid(filtered_group, LAT_COL, LON_COL, ANGLE_COL)

                results_list.append({
                    'Radar_ID': radar_id, 'Obs_Count': len(full_group), 'Filtered_Count': len(filtered_group),
                    'Avg_Freq': avg_params[FREQ_COL], 'Avg_PRI': avg_params[PRI_COL], 'Avg_PW': avg_params[PW_COL],
                    'Est_Lat_All': est_lat_all, 'Est_Lon_All': est_lon_all,
                    'Est_Lat_Filtered': est_lat_filtered, 'Est_Lon_Filtered': est_lon_filtered
                })

        # --- Display Final Report ---
        print("\n-------------------------------------------")
        print(f"Final Report for {INPUT_FILE}")
        print("-------------------------------------------")
        results_df = pd.DataFrame(results_list)
        pd.set_option('display.width', 120)
        pd.set_option('display.max_columns', 11)
        pd.set_option('display.float_format', '{:.4f}'.format)
        print(results_df.to_string())

        output_filename = f"results_dbscan_time_{INPUT_FILE.split('.')[0]}.csv"
        results_df.to_csv(output_filename, index=False)
        print(f"\nResults have also been saved to '{output_filename}'")

if __name__ == "__main__":
    run_analysis()

Loading observations from Observations_011.xlsx...
Converting 'Time' column from time objects to seconds...

--- PART 2: Running Final Analysis with eps = 0.55 ---

--- Clustering Results ---
 1    1222
 0     510
 2     490
-1      85
Name: count, dtype: int64
Found 3 distinct radars.

Calculating parameters and estimating locations...

-------------------------------------------
Final Report for Observations_011.xlsx
-------------------------------------------
   Radar_ID  Obs_Count  Filtered_Count  Avg_Freq   Avg_PRI  Avg_PW  Est_Lat_All  Est_Lon_All  Est_Lat_Filtered  Est_Lon_Filtered
0    0.0000        510             510  876.7000 2099.7980  1.5000      -3.8306      19.8833           -3.8306           19.8833
1    1.0000       1222            1222 1498.8854   56.9255  1.0841      -4.0608      20.0509           -4.0608           20.0509
2    2.0000        490             490  877.3571 2098.6429  1.9400      -3.7219      19.8745           -3.7219           19.8745

Results have als

**Observation_011 with removing DF_Q[1,2]**
*No difference, as there is no data values of 1 and 2 in the dataset*

In [28]:
def cross_track_distance(point_lat, point_lon, start_lat, start_lon, bearing_deg):
    """
    Calculates the shortest distance (in meters) from a point to a great-circle
    path (a line of bearing), using numerically stable formulas.
    """
    R_earth = 6371000  # Earth radius in meters
    bearing_rad = np.radians(bearing_deg)

    # Convert all degree values to radians for trigonometric functions
    lat1, lon1 = np.radians(start_lat), np.radians(start_lon)
    lat3, lon3 = np.radians(point_lat), np.radians(point_lon)

    # Calculate angular distance between the aircraft and the candidate point
    arccos_input = np.sin(lat1) * np.sin(lat3) + np.cos(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    ang_dist_ac = np.arccos(np.clip(arccos_input, -1.0, 1.0)) # Clip for numerical stability

    # Calculate the bearing from the aircraft to the candidate point
    bearing_ac_rad = np.arctan2(
        np.sin(lon3 - lon1) * np.cos(lat3),
        np.cos(lat1) * np.sin(lat3) - np.sin(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    )

    delta_bearing = bearing_ac_rad - bearing_rad

    # Calculate the final cross-track distance
    arcsin_input = np.sin(ang_dist_ac) * np.sin(delta_bearing)
    d_xt = np.arcsin(np.clip(arcsin_input, -1.0, 1.0)) # Clip for numerical stability

    return R_earth * d_xt

def estimate_location_scipy(df_group, lat_col, lon_col, angle_col):
    """
    Finds the optimal emitter location using a non-linear solver (SciPy).
    This is the high-accuracy primary method.
    """
    if len(df_group) < 2: return None, None

    # Define an error function to minimize: the sum of squared distances to all lines of bearing.
    def error_function(x):
        candidate_lat, candidate_lon = x[0], x[1]
        distances = df_group.apply(lambda row: cross_track_distance(candidate_lat, candidate_lon, row[lat_col], row[lon_col], row[angle_col]), axis=1)
        return np.sum(distances**2)

    # Use the average location of observations as a smart initial guess
    initial_guess = [df_group[lat_col].mean(), df_group[lon_col].mean()]

    # Run the optimization solver
    result = minimize(error_function, initial_guess, method='Nelder-Mead')

    return (result.x[0], result.x[1]) if result.success else (None, None)

def estimate_location_utm(df_group, lat_col, lon_col, angle_col):
    """
    Estimates emitter location using a linear least-squares method on a UTM projection.
    This serves as a robust fallback method.
    """
    if len(df_group) < 2: return None, None

    # Determine the correct UTM zone from the average longitude
    avg_lon = df_group[lon_col].mean()
    utm_zone = int((avg_lon + 180) / 6) + 1
    wgs84, utm_crs = pyproj.CRS("EPSG:4326"), pyproj.CRS(f"EPSG:326{utm_zone}")

    # Convert observation coordinates to the more accurate UTM grid
    transformer_to_utm = pyproj.Transformer.from_crs(wgs84, utm_crs, always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(df_group[lon_col].values, df_group[lat_col].values)

    # Formulate and solve the linear system of equations
    angle_rad = np.radians(90 - df_group[angle_col])
    A = np.vstack([np.cos(angle_rad), np.sin(angle_rad)]).T
    b = np.sum(A * np.vstack([utm_x, utm_y]).T, axis=1)
    try:
        loc, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        # Convert the UTM result back to standard latitude/longitude
        transformer_to_wgs84 = pyproj.Transformer.from_crs(utm_crs, wgs84, always_xy=True)
        est_lon, est_lat = transformer_to_wgs84.transform(loc[0], loc[1])
        return est_lat, est_lon
    except np.linalg.LinAlgError:
        return None, None

def estimate_location_hybrid(df_group, lat_col, lon_col, angle_col):
    """
    Manages the location estimation, trying the high-accuracy SciPy method first
    and falling back to the UTM method if it fails.
    """
    est_lat, est_lon = estimate_location_scipy(df_group, lat_col, lon_col, angle_col)
    if est_lat is None:
        # This fallback makes the solution robust to poor signal geometry
        est_lat, est_lon = estimate_location_utm(df_group, lat_col, lon_col, angle_col)
    return est_lat, est_lon

def run_analysis():
    """Main function to run the full analysis workflow."""

    # --- Configuration ---
    INPUT_FILE = 'Observations_011.xlsx'
    # The user updates this value after the first run
    CHOSEN_EPS = 0.55
    MIN_SAMPLES = 20
    FILTER_LIST = []
    # --- Define Column Names ---
    FREQ_COL, PRI_COL, PW_COL, TIME_COL = 'Freq', 'PRI', 'PW', 'Time'
    LAT_COL, LON_COL, ANGLE_COL, DFQ_COL = 'Lat', 'Lon', 'Angle', 'DF_Q'

    # --- Load and Scale Data ---
    print(f"Loading observations from {INPUT_FILE}...")
    obs_df = pd.read_excel(INPUT_FILE)

    # Convert time objects to a numerical format (seconds since midnight)
    def time_to_seconds(t):
        if isinstance(t, datetime.time):
            return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
        return np.nan
    if obs_df[TIME_COL].dtype == 'object':
        print("Converting 'Time' column from time objects to seconds...")
        obs_df[TIME_COL] = obs_df[TIME_COL].apply(time_to_seconds)

    obs_df[TIME_COL] = pd.to_numeric(obs_df[TIME_COL], errors='coerce')
    obs_df.dropna(subset=[TIME_COL], inplace=True)

    # Select and scale the four features for clustering
    features = obs_df[[FREQ_COL, PRI_COL, PW_COL, TIME_COL]]
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # --- Workflow Controller ---
    if CHOSEN_EPS == 0.0:
        # --- PART 1: Generate the `eps` vs. Number of Clusters Plot ---
        print("\n--- PART 1: Generating `eps` vs. Number of Clusters Plot ---")
        eps_range = np.arange(0.1, 1.01, 0.01)
        cluster_counts = []

        print("Testing a range of `eps` values to generate the diagnostic plot...")
        for eps in eps_range:
            dbscan = DBSCAN(eps=eps, min_samples=MIN_SAMPLES)
            clusters = dbscan.fit_predict(features_scaled)
            num_clusters = len(set(clusters) - {-1})
            cluster_counts.append(num_clusters)

        # Use spline interpolation to create a smooth curve for better visualization
        if len(eps_range) > 3:
            X_Y_Spline = make_interp_spline(eps_range, cluster_counts)
            X_ = np.linspace(eps_range.min(), eps_range.max(), 500)
            Y_ = X_Y_Spline(X_)
        else:
            X_ = eps_range
            Y_ = cluster_counts

        plt.figure(figsize=(12, 7))
        plt.plot(X_, Y_, label='Smoothed Trend')
        plt.scatter(eps_range, cluster_counts, color='red', zorder=5, s=10, label='Actual Data Points')
        plt.xticks(np.arange(0, 1.01, 0.05), rotation=90)
        plt.xlabel("Epsilon (`eps`) Value")
        plt.ylabel("Number of Clusters Found")
        plt.title(f"`eps` vs. Number of Clusters (for min_samples={MIN_SAMPLES})")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig("eps_vs_clusters_plot.png")

        print("\nACTION REQUIRED:")
        print("A plot named 'eps_vs_clusters_plot.png' has been saved.")
        print("1. Open the plot and find the 'elbow' or a stable 'plateau' region.")
        print("2. Choose an `eps` value from the start of this stable region.")
        print(f"3. Update the 'CHOSEN_EPS' variable in this script from {CHOSEN_EPS} to your new value.")
        print("4. Run the script again to get your final analysis.\n")
    else:
        # --- PART 2: Perform Final Analysis with Chosen EPS ---
        print(f"\n--- PART 2: Running Final Analysis with eps = {CHOSEN_EPS} ---")
        dbscan = DBSCAN(eps=CHOSEN_EPS, min_samples=MIN_SAMPLES)
        clusters = dbscan.fit_predict(features_scaled)
        obs_df.loc[features.index, 'Radar_ID'] = clusters

        print("\n--- Clustering Results ---")
        print(pd.Series(clusters).value_counts())
        obs_clustered_df = obs_df[obs_df['Radar_ID'] != -1].copy()
        num_radars_found = len(obs_clustered_df['Radar_ID'].unique())
        print(f"Found {num_radars_found} distinct radars.")

        results_list = []
        if num_radars_found > 0:
            print("\nCalculating parameters and estimating locations...")
            for radar_id in sorted(obs_clustered_df['Radar_ID'].unique()):
                full_group = obs_clustered_df[obs_clustered_df['Radar_ID'] == radar_id]
                filtered_group = full_group[~full_group[DFQ_COL].isin(FILTER_LIST)]
                avg_params = full_group[[FREQ_COL, PRI_COL, PW_COL]].mean()

                est_lat_all, est_lon_all = estimate_location_hybrid(full_group, LAT_COL, LON_COL, ANGLE_COL)
                est_lat_filtered, est_lon_filtered = estimate_location_hybrid(filtered_group, LAT_COL, LON_COL, ANGLE_COL)

                results_list.append({
                    'Radar_ID': radar_id, 'Obs_Count': len(full_group), 'Filtered_Count': len(filtered_group),
                    'Avg_Freq': avg_params[FREQ_COL], 'Avg_PRI': avg_params[PRI_COL], 'Avg_PW': avg_params[PW_COL],
                    'Est_Lat_All': est_lat_all, 'Est_Lon_All': est_lon_all,
                    'Est_Lat_Filtered': est_lat_filtered, 'Est_Lon_Filtered': est_lon_filtered
                })

        # --- Display Final Report ---
        print("\n-------------------------------------------")
        print(f"Final Report for {INPUT_FILE}")
        print("-------------------------------------------")
        results_df = pd.DataFrame(results_list)
        pd.set_option('display.width', 120)
        pd.set_option('display.max_columns', 11)
        pd.set_option('display.float_format', '{:.4f}'.format)
        print(results_df.to_string())

        output_filename = f"results_dbscan_time_{INPUT_FILE.split('.')[0]}.csv"
        results_df.to_csv(output_filename, index=False)
        print(f"\nResults have also been saved to '{output_filename}'")

if __name__ == "__main__":
    run_analysis()

Loading observations from Observations_011.xlsx...
Converting 'Time' column from time objects to seconds...

--- PART 2: Running Final Analysis with eps = 0.55 ---

--- Clustering Results ---
 1    1222
 0     510
 2     490
-1      85
Name: count, dtype: int64
Found 3 distinct radars.

Calculating parameters and estimating locations...

-------------------------------------------
Final Report for Observations_011.xlsx
-------------------------------------------
   Radar_ID  Obs_Count  Filtered_Count  Avg_Freq   Avg_PRI  Avg_PW  Est_Lat_All  Est_Lon_All  Est_Lat_Filtered  Est_Lon_Filtered
0    0.0000        510             510  876.7000 2099.7980  1.5000      -3.8306      19.8833           -3.8306           19.8833
1    1.0000       1222            1222 1498.8854   56.9255  1.0841      -4.0608      20.0509           -4.0608           20.0509
2    2.0000        490             490  877.3571 2098.6429  1.9400      -3.7219      19.8745           -3.7219           19.8745

Results have als

**Observation_012 without removing DF_Q[1,2]**

In [30]:
def cross_track_distance(point_lat, point_lon, start_lat, start_lon, bearing_deg):
    """
    Calculates the shortest distance (in meters) from a point to a great-circle
    path (a line of bearing), using numerically stable formulas.
    """
    R_earth = 6371000  # Earth radius in meters
    bearing_rad = np.radians(bearing_deg)

    # Convert all degree values to radians for trigonometric functions
    lat1, lon1 = np.radians(start_lat), np.radians(start_lon)
    lat3, lon3 = np.radians(point_lat), np.radians(point_lon)

    # Calculate angular distance between the aircraft and the candidate point
    arccos_input = np.sin(lat1) * np.sin(lat3) + np.cos(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    ang_dist_ac = np.arccos(np.clip(arccos_input, -1.0, 1.0)) # Clip for numerical stability

    # Calculate the bearing from the aircraft to the candidate point
    bearing_ac_rad = np.arctan2(
        np.sin(lon3 - lon1) * np.cos(lat3),
        np.cos(lat1) * np.sin(lat3) - np.sin(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    )

    delta_bearing = bearing_ac_rad - bearing_rad

    # Calculate the final cross-track distance
    arcsin_input = np.sin(ang_dist_ac) * np.sin(delta_bearing)
    d_xt = np.arcsin(np.clip(arcsin_input, -1.0, 1.0)) # Clip for numerical stability

    return R_earth * d_xt

def estimate_location_scipy(df_group, lat_col, lon_col, angle_col):
    """
    Finds the optimal emitter location using a non-linear solver (SciPy).
    This is the high-accuracy primary method.
    """
    if len(df_group) < 2: return None, None

    # Define an error function to minimize: the sum of squared distances to all lines of bearing.
    def error_function(x):
        candidate_lat, candidate_lon = x[0], x[1]
        distances = df_group.apply(lambda row: cross_track_distance(candidate_lat, candidate_lon, row[lat_col], row[lon_col], row[angle_col]), axis=1)
        return np.sum(distances**2)

    # Use the average location of observations as a smart initial guess
    initial_guess = [df_group[lat_col].mean(), df_group[lon_col].mean()]

    # Run the optimization solver
    result = minimize(error_function, initial_guess, method='Nelder-Mead')

    return (result.x[0], result.x[1]) if result.success else (None, None)

def estimate_location_utm(df_group, lat_col, lon_col, angle_col):
    """
    Estimates emitter location using a linear least-squares method on a UTM projection.
    This serves as a robust fallback method.
    """
    if len(df_group) < 2: return None, None

    # Determine the correct UTM zone from the average longitude
    avg_lon = df_group[lon_col].mean()
    utm_zone = int((avg_lon + 180) / 6) + 1
    wgs84, utm_crs = pyproj.CRS("EPSG:4326"), pyproj.CRS(f"EPSG:326{utm_zone}")

    # Convert observation coordinates to the more accurate UTM grid
    transformer_to_utm = pyproj.Transformer.from_crs(wgs84, utm_crs, always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(df_group[lon_col].values, df_group[lat_col].values)

    # Formulate and solve the linear system of equations
    angle_rad = np.radians(90 - df_group[angle_col])
    A = np.vstack([np.cos(angle_rad), np.sin(angle_rad)]).T
    b = np.sum(A * np.vstack([utm_x, utm_y]).T, axis=1)
    try:
        loc, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        # Convert the UTM result back to standard latitude/longitude
        transformer_to_wgs84 = pyproj.Transformer.from_crs(utm_crs, wgs84, always_xy=True)
        est_lon, est_lat = transformer_to_wgs84.transform(loc[0], loc[1])
        return est_lat, est_lon
    except np.linalg.LinAlgError:
        return None, None

def estimate_location_hybrid(df_group, lat_col, lon_col, angle_col):
    """
    Manages the location estimation, trying the high-accuracy SciPy method first
    and falling back to the UTM method if it fails.
    """
    est_lat, est_lon = estimate_location_scipy(df_group, lat_col, lon_col, angle_col)
    if est_lat is None:
        # This fallback makes the solution robust to poor signal geometry
        est_lat, est_lon = estimate_location_utm(df_group, lat_col, lon_col, angle_col)
    return est_lat, est_lon

def run_analysis():
    """Main function to run the full analysis workflow."""

    # --- Configuration ---
    INPUT_FILE = 'Observations_012.xlsx'
    # The user updates this value after the first run
    CHOSEN_EPS = 0.42
    MIN_SAMPLES = 20
    FILTER_LIST = [1,2]
    # --- Define Column Names ---
    FREQ_COL, PRI_COL, PW_COL, TIME_COL = 'Freq', 'PRI', 'PW', 'Time'
    LAT_COL, LON_COL, ANGLE_COL, DFQ_COL = 'Lat', 'Lon', 'Angle', 'DF_Q'

    # --- Load and Scale Data ---
    print(f"Loading observations from {INPUT_FILE}...")
    obs_df = pd.read_excel(INPUT_FILE)

    # Convert time objects to a numerical format (seconds since midnight)
    def time_to_seconds(t):
        if isinstance(t, datetime.time):
            return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
        return np.nan
    if obs_df[TIME_COL].dtype == 'object':
        print("Converting 'Time' column from time objects to seconds...")
        obs_df[TIME_COL] = obs_df[TIME_COL].apply(time_to_seconds)

    obs_df[TIME_COL] = pd.to_numeric(obs_df[TIME_COL], errors='coerce')
    obs_df.dropna(subset=[TIME_COL], inplace=True)

    # Select and scale the four features for clustering
    features = obs_df[[FREQ_COL, PRI_COL, PW_COL, TIME_COL]]
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # --- Workflow Controller ---
    if CHOSEN_EPS == 0.0:
        # --- PART 1: Generate the `eps` vs. Number of Clusters Plot ---
        print("\n--- PART 1: Generating `eps` vs. Number of Clusters Plot ---")
        eps_range = np.arange(0.1, 1.01, 0.01)
        cluster_counts = []

        print("Testing a range of `eps` values to generate the diagnostic plot...")
        for eps in eps_range:
            dbscan = DBSCAN(eps=eps, min_samples=MIN_SAMPLES)
            clusters = dbscan.fit_predict(features_scaled)
            num_clusters = len(set(clusters) - {-1})
            cluster_counts.append(num_clusters)

        # Use spline interpolation to create a smooth curve for better visualization
        if len(eps_range) > 3:
            X_Y_Spline = make_interp_spline(eps_range, cluster_counts)
            X_ = np.linspace(eps_range.min(), eps_range.max(), 500)
            Y_ = X_Y_Spline(X_)
        else:
            X_ = eps_range
            Y_ = cluster_counts

        plt.figure(figsize=(12, 7))
        plt.plot(X_, Y_, label='Smoothed Trend')
        plt.scatter(eps_range, cluster_counts, color='red', zorder=5, s=10, label='Actual Data Points')
        plt.xticks(np.arange(0, 1.01, 0.05), rotation=90)
        plt.xlabel("Epsilon (`eps`) Value")
        plt.ylabel("Number of Clusters Found")
        plt.title(f"`eps` vs. Number of Clusters (for min_samples={MIN_SAMPLES})")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig("eps_vs_clusters_plot.png")

        print("\nACTION REQUIRED:")
        print("A plot named 'eps_vs_clusters_plot.png' has been saved.")
        print("1. Open the plot and find the 'elbow' or a stable 'plateau' region.")
        print("2. Choose an `eps` value from the start of this stable region.")
        print(f"3. Update the 'CHOSEN_EPS' variable in this script from {CHOSEN_EPS} to your new value.")
        print("4. Run the script again to get your final analysis.\n")
    else:
        # --- PART 2: Perform Final Analysis with Chosen EPS ---
        print(f"\n--- PART 2: Running Final Analysis with eps = {CHOSEN_EPS} ---")
        dbscan = DBSCAN(eps=CHOSEN_EPS, min_samples=MIN_SAMPLES)
        clusters = dbscan.fit_predict(features_scaled)
        obs_df.loc[features.index, 'Radar_ID'] = clusters

        print("\n--- Clustering Results ---")
        print(pd.Series(clusters).value_counts())
        obs_clustered_df = obs_df[obs_df['Radar_ID'] != -1].copy()
        num_radars_found = len(obs_clustered_df['Radar_ID'].unique())
        print(f"Found {num_radars_found} distinct radars.")

        results_list = []
        if num_radars_found > 0:
            print("\nCalculating parameters and estimating locations...")
            for radar_id in sorted(obs_clustered_df['Radar_ID'].unique()):
                full_group = obs_clustered_df[obs_clustered_df['Radar_ID'] == radar_id]
                filtered_group = full_group[~full_group[DFQ_COL].isin(FILTER_LIST)]
                avg_params = full_group[[FREQ_COL, PRI_COL, PW_COL]].mean()

                est_lat_all, est_lon_all = estimate_location_hybrid(full_group, LAT_COL, LON_COL, ANGLE_COL)
                est_lat_filtered, est_lon_filtered = estimate_location_hybrid(filtered_group, LAT_COL, LON_COL, ANGLE_COL)

                results_list.append({
                    'Radar_ID': radar_id, 'Obs_Count': len(full_group), 'Filtered_Count': len(filtered_group),
                    'Avg_Freq': avg_params[FREQ_COL], 'Avg_PRI': avg_params[PRI_COL], 'Avg_PW': avg_params[PW_COL],
                    'Est_Lat_All': est_lat_all, 'Est_Lon_All': est_lon_all,
                    'Est_Lat_Filtered': est_lat_filtered, 'Est_Lon_Filtered': est_lon_filtered
                })

        # --- Display Final Report ---
        print("\n-------------------------------------------")
        print(f"Final Report for {INPUT_FILE}")
        print("-------------------------------------------")
        results_df = pd.DataFrame(results_list)
        pd.set_option('display.width', 120)
        pd.set_option('display.max_columns', 11)
        pd.set_option('display.float_format', '{:.4f}'.format)
        print(results_df.to_string())

        output_filename = f"results_dbscan_time_{INPUT_FILE.split('.')[0]}.csv"
        results_df.to_csv(output_filename, index=False)
        print(f"\nResults have also been saved to '{output_filename}'")

if __name__ == "__main__":
    run_analysis()

Loading observations from Observations_012.xlsx...
Converting 'Time' column from time objects to seconds...

--- PART 2: Running Final Analysis with eps = 0.42 ---

--- Clustering Results ---
 1    7346
 0    3678
-1       9
Name: count, dtype: int64
Found 2 distinct radars.

Calculating parameters and estimating locations...

-------------------------------------------
Final Report for Observations_012.xlsx
-------------------------------------------
   Radar_ID  Obs_Count  Filtered_Count  Avg_Freq   Avg_PRI   Avg_PW  Est_Lat_All  Est_Lon_All  Est_Lat_Filtered  Est_Lon_Filtered
0    0.0000       3678            3678 1057.6158 2376.8839 381.9182      -3.7402      19.9336           -3.7402           19.9336
1    1.0000       7346            7346 3141.1909 2263.9838   8.7161      -2.2517      19.9757           -2.2517           19.9757

Results have also been saved to 'results_dbscan_time_Observations_012.csv'


**Observation_012 with removing DF_Q[1,2]**
*Again, No difference, as there is no data values of 1 and 2 in the dataset*

In [34]:
def cross_track_distance(point_lat, point_lon, start_lat, start_lon, bearing_deg):
    """
    Calculates the shortest distance (in meters) from a point to a great-circle
    path (a line of bearing), using numerically stable formulas.
    """
    R_earth = 6371000  # Earth radius in meters
    bearing_rad = np.radians(bearing_deg)

    # Convert all degree values to radians for trigonometric functions
    lat1, lon1 = np.radians(start_lat), np.radians(start_lon)
    lat3, lon3 = np.radians(point_lat), np.radians(point_lon)

    # Calculate angular distance between the aircraft and the candidate point
    arccos_input = np.sin(lat1) * np.sin(lat3) + np.cos(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    ang_dist_ac = np.arccos(np.clip(arccos_input, -1.0, 1.0)) # Clip for numerical stability

    # Calculate the bearing from the aircraft to the candidate point
    bearing_ac_rad = np.arctan2(
        np.sin(lon3 - lon1) * np.cos(lat3),
        np.cos(lat1) * np.sin(lat3) - np.sin(lat1) * np.cos(lat3) * np.cos(lon3 - lon1)
    )

    delta_bearing = bearing_ac_rad - bearing_rad

    # Calculate the final cross-track distance
    arcsin_input = np.sin(ang_dist_ac) * np.sin(delta_bearing)
    d_xt = np.arcsin(np.clip(arcsin_input, -1.0, 1.0)) # Clip for numerical stability

    return R_earth * d_xt

def estimate_location_scipy(df_group, lat_col, lon_col, angle_col):
    """
    Finds the optimal emitter location using a non-linear solver (SciPy).
    This is the high-accuracy primary method.
    """
    if len(df_group) < 2: return None, None

    # Define an error function to minimize: the sum of squared distances to all lines of bearing.
    def error_function(x):
        candidate_lat, candidate_lon = x[0], x[1]
        distances = df_group.apply(lambda row: cross_track_distance(candidate_lat, candidate_lon, row[lat_col], row[lon_col], row[angle_col]), axis=1)
        return np.sum(distances**2)

    # Use the average location of observations as a smart initial guess
    initial_guess = [df_group[lat_col].mean(), df_group[lon_col].mean()]

    # Run the optimization solver
    result = minimize(error_function, initial_guess, method='Nelder-Mead')

    return (result.x[0], result.x[1]) if result.success else (None, None)

def estimate_location_utm(df_group, lat_col, lon_col, angle_col):
    """
    Estimates emitter location using a linear least-squares method on a UTM projection.
    This serves as a robust fallback method.
    """
    if len(df_group) < 2: return None, None

    # Determine the correct UTM zone from the average longitude
    avg_lon = df_group[lon_col].mean()
    utm_zone = int((avg_lon + 180) / 6) + 1
    wgs84, utm_crs = pyproj.CRS("EPSG:4326"), pyproj.CRS(f"EPSG:326{utm_zone}")

    # Convert observation coordinates to the more accurate UTM grid
    transformer_to_utm = pyproj.Transformer.from_crs(wgs84, utm_crs, always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(df_group[lon_col].values, df_group[lat_col].values)

    # Formulate and solve the linear system of equations
    angle_rad = np.radians(90 - df_group[angle_col])
    A = np.vstack([np.cos(angle_rad), np.sin(angle_rad)]).T
    b = np.sum(A * np.vstack([utm_x, utm_y]).T, axis=1)
    try:
        loc, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        # Convert the UTM result back to standard latitude/longitude
        transformer_to_wgs84 = pyproj.Transformer.from_crs(utm_crs, wgs84, always_xy=True)
        est_lon, est_lat = transformer_to_wgs84.transform(loc[0], loc[1])
        return est_lat, est_lon
    except np.linalg.LinAlgError:
        return None, None

def estimate_location_hybrid(df_group, lat_col, lon_col, angle_col):
    """
    Manages the location estimation, trying the high-accuracy SciPy method first
    and falling back to the UTM method if it fails.
    """
    est_lat, est_lon = estimate_location_scipy(df_group, lat_col, lon_col, angle_col)
    if est_lat is None:
        # This fallback makes the solution robust to poor signal geometry
        est_lat, est_lon = estimate_location_utm(df_group, lat_col, lon_col, angle_col)
    return est_lat, est_lon

def run_analysis():
    """Main function to run the full analysis workflow."""

    # --- Configuration ---
    INPUT_FILE = 'Observations_012.xlsx'
    # The user updates this value after the first run
    CHOSEN_EPS = 0.42
    MIN_SAMPLES = 20
    FILTER_LIST = []
    # --- Define Column Names ---
    FREQ_COL, PRI_COL, PW_COL, TIME_COL = 'Freq', 'PRI', 'PW', 'Time'
    LAT_COL, LON_COL, ANGLE_COL, DFQ_COL = 'Lat', 'Lon', 'Angle', 'DF_Q'

    # --- Load and Scale Data ---
    print(f"Loading observations from {INPUT_FILE}...")
    obs_df = pd.read_excel(INPUT_FILE)

    # Convert time objects to a numerical format (seconds since midnight)
    def time_to_seconds(t):
        if isinstance(t, datetime.time):
            return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6
        return np.nan
    if obs_df[TIME_COL].dtype == 'object':
        print("Converting 'Time' column from time objects to seconds...")
        obs_df[TIME_COL] = obs_df[TIME_COL].apply(time_to_seconds)

    obs_df[TIME_COL] = pd.to_numeric(obs_df[TIME_COL], errors='coerce')
    obs_df.dropna(subset=[TIME_COL], inplace=True)

    # Select and scale the four features for clustering
    features = obs_df[[FREQ_COL, PRI_COL, PW_COL, TIME_COL]]
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # --- Workflow Controller ---
    if CHOSEN_EPS == 0.0:
        # --- PART 1: Generate the `eps` vs. Number of Clusters Plot ---
        print("\n--- PART 1: Generating `eps` vs. Number of Clusters Plot ---")
        eps_range = np.arange(0.1, 1.01, 0.01)
        cluster_counts = []

        print("Testing a range of `eps` values to generate the diagnostic plot...")
        for eps in eps_range:
            dbscan = DBSCAN(eps=eps, min_samples=MIN_SAMPLES)
            clusters = dbscan.fit_predict(features_scaled)
            num_clusters = len(set(clusters) - {-1})
            cluster_counts.append(num_clusters)

        # Use spline interpolation to create a smooth curve for better visualization
        if len(eps_range) > 3:
            X_Y_Spline = make_interp_spline(eps_range, cluster_counts)
            X_ = np.linspace(eps_range.min(), eps_range.max(), 500)
            Y_ = X_Y_Spline(X_)
        else:
            X_ = eps_range
            Y_ = cluster_counts

        plt.figure(figsize=(12, 7))
        plt.plot(X_, Y_, label='Smoothed Trend')
        plt.scatter(eps_range, cluster_counts, color='red', zorder=5, s=10, label='Actual Data Points')
        plt.xticks(np.arange(0, 1.01, 0.05), rotation=90)
        plt.xlabel("Epsilon (`eps`) Value")
        plt.ylabel("Number of Clusters Found")
        plt.title(f"`eps` vs. Number of Clusters (for min_samples={MIN_SAMPLES})")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig("eps_vs_clusters_plot.png")

        print("\nACTION REQUIRED:")
        print("A plot named 'eps_vs_clusters_plot.png' has been saved.")
        print("1. Open the plot and find the 'elbow' or a stable 'plateau' region.")
        print("2. Choose an `eps` value from the start of this stable region.")
        print(f"3. Update the 'CHOSEN_EPS' variable in this script from {CHOSEN_EPS} to your new value.")
        print("4. Run the script again to get your final analysis.\n")
    else:
        # --- PART 2: Perform Final Analysis with Chosen EPS ---
        print(f"\n--- PART 2: Running Final Analysis with eps = {CHOSEN_EPS} ---")
        dbscan = DBSCAN(eps=CHOSEN_EPS, min_samples=MIN_SAMPLES)
        clusters = dbscan.fit_predict(features_scaled)
        obs_df.loc[features.index, 'Radar_ID'] = clusters

        print("\n--- Clustering Results ---")
        print(pd.Series(clusters).value_counts())
        obs_clustered_df = obs_df[obs_df['Radar_ID'] != -1].copy()
        num_radars_found = len(obs_clustered_df['Radar_ID'].unique())
        print(f"Found {num_radars_found} distinct radars.")

        results_list = []
        if num_radars_found > 0:
            print("\nCalculating parameters and estimating locations...")
            for radar_id in sorted(obs_clustered_df['Radar_ID'].unique()):
                full_group = obs_clustered_df[obs_clustered_df['Radar_ID'] == radar_id]
                filtered_group = full_group[~full_group[DFQ_COL].isin(FILTER_LIST)]
                avg_params = full_group[[FREQ_COL, PRI_COL, PW_COL]].mean()

                est_lat_all, est_lon_all = estimate_location_hybrid(full_group, LAT_COL, LON_COL, ANGLE_COL)
                est_lat_filtered, est_lon_filtered = estimate_location_hybrid(filtered_group, LAT_COL, LON_COL, ANGLE_COL)

                results_list.append({
                    'Radar_ID': radar_id, 'Obs_Count': len(full_group), 'Filtered_Count': len(filtered_group),
                    'Avg_Freq': avg_params[FREQ_COL], 'Avg_PRI': avg_params[PRI_COL], 'Avg_PW': avg_params[PW_COL],
                    'Est_Lat_All': est_lat_all, 'Est_Lon_All': est_lon_all,
                    'Est_Lat_Filtered': est_lat_filtered, 'Est_Lon_Filtered': est_lon_filtered
                })

        # --- Display Final Report ---
        print("\n-------------------------------------------")
        print(f"Final Report for {INPUT_FILE}")
        print("-------------------------------------------")
        results_df = pd.DataFrame(results_list)
        pd.set_option('display.width', 120)
        pd.set_option('display.max_columns', 11)
        pd.set_option('display.float_format', '{:.4f}'.format)
        print(results_df.to_string())

        output_filename = f"results_dbscan_time_{INPUT_FILE.split('.')[0]}.csv"
        results_df.to_csv(output_filename, index=False)
        print(f"\nResults have also been saved to '{output_filename}'")

if __name__ == "__main__":
    run_analysis()

Loading observations from Observations_012.xlsx...
Converting 'Time' column from time objects to seconds...

--- PART 2: Running Final Analysis with eps = 0.42 ---

--- Clustering Results ---
 1    7346
 0    3678
-1       9
Name: count, dtype: int64
Found 2 distinct radars.

Calculating parameters and estimating locations...

-------------------------------------------
Final Report for Observations_012.xlsx
-------------------------------------------
   Radar_ID  Obs_Count  Filtered_Count  Avg_Freq   Avg_PRI   Avg_PW  Est_Lat_All  Est_Lon_All  Est_Lat_Filtered  Est_Lon_Filtered
0    0.0000       3678            3678 1057.6158 2376.8839 381.9182      -3.7402      19.9336           -3.7402           19.9336
1    1.0000       7346            7346 3141.1909 2263.9838   8.7161      -2.2517      19.9757           -2.2517           19.9757

Results have also been saved to 'results_dbscan_time_Observations_012.csv'
